# Study regions: glacier location maps (Fig. 1 and Fig. A3b)

This notebook makes the location maps of the glaciers with point mass balance (PMB) measurements:

| Paper figure | Region | Section | Saved as (in `figures/paperTF/`) |
|---|---|---|---|
| Fig. 1a | Alaska (RGI 01) | Alaska | `fig1_a_maps_ALA_gl.pdf` |
| Fig. 1b | Svalbard (RGI 07) | Svalbard | `fig1_b_maps_SJM_gl.pdf` |
| Fig. 1c | Norway (RGI 08) | NOR | `fig1_c_maps_NOR_gl.pdf` |
| Fig. 1d | Iceland (RGI 06) | Iceland | `fig1_d_maps_ISL_gl.pdf` |
| Fig. 1e | Central European Alps (RGI 11), coloured by country | CEU | `fig1_e_maps_CEU_gl.pdf` + `fig1_e_maps_CEU_gl_legend.pdf` |
| Fig. 1f | Central Asia (RGI 13), coloured by RGI sub-region | Central Asia | `fig1_f_maps_CA_gl.pdf` + `fig1_f_maps_CA_legend_color.pdf` |
| Fig. A3b | Test glaciers of Central Asia sub-region 3 | Appendix A3 (b) | `appendix/A3_b.png` |

The panels are combined into Fig. 1 and Fig. A3 outside the code. Each map also prints the number of glaciers and the range of years, which are the numbers quoted in the Fig. 1 caption.

**Inputs** (under `cfg.dataPath`): the stake datasets written by the `PMB_sources` notebooks (read with `load_stakes_for_rgi_region`) and the RGI v6 outlines in `RGI_v6/<region>/`. The A3 (b) map uses the five test glaciers of the Central Asia sub-region 3 split from `2.2 Within_region`, listed in the notebook.

## Imports & setting up:

In [ ]:
# Silence KeOps (used by geomloss for Sinkhorn distances) before it is imported
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import warnings
import pandas as pd
import massbalancemachine as mbm

# Shared config (NATURE_PALETTE, IT_GLACIERS, AT_GLACIERS, ...)
from papers.Transformer_TF.scripts.config_TF_Europe import *
# Per-glacier info table of one RGI region (location, number of measurements, sub-region)
from papers.Transformer_TF.scripts.utils.glaciers import build_region_glacier_info
# Glacier location map used for Fig. 1 and Fig. A3b, and the Nature figure specs
from papers.Transformer_TF.scripts.plotting.study_region_maps import plot_glacier_locations_map
from papers.Transformer_TF.scripts.plotting.style import NATURE_SPECS

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Call config file (data paths, seed, etc.)
cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)  # reproducibility
mbm.plots.use_mbm_style()

## Maps
Each section below uses two helper functions from `scripts/`:
- `build_region_glacier_info` (`scripts/utils/glaciers.py`): loads the stakes of one RGI region and returns one row per glacier (mean stake location, number of measurements, measurements per period, `SOURCE_CODE` and `O2Region`). For Central Asia and Alaska it adds the RGI O2 sub-region; for Central Europe it labels the Italian and Austrian glaciers.
- `plot_glacier_locations_map` (`scripts/plotting/study_region_maps.py`): plots the glaciers on a Cartopy map over the RGI outlines, optionally coloured by a category (`hue_col`) and with the legend in a separate figure (`legend_separate=True`). It also prints the number of glaciers and the range of years.

All maps use a fixed marker size (`use_size=False`), so they show locations only.

### (e) Central European Alps (CEU), Fig. 1e:
Glaciers coloured by country (CH, FR, IT, AT). The legend is saved separately.

In [ ]:
# One row per glacier of RGI 11; IT/AT labels are set inside build_region_glacier_info
data_CEU, glacier_outline_CEU, glacier_df_CEU = build_region_glacier_info(
    cfg,
    rgi_region_id="11",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
)
# IT/AT override now happens automatically inside the function

# Colour per country (SOURCE_CODE)
palette_ceu = {
    "CH": NATURE_PALETTE["blue"],
    "FR": NATURE_PALETTE["orange"],
    "IT": NATURE_PALETTE["bluish_green"],
    "AT": NATURE_PALETTE["reddish_purple"],
}

fig, ax, glacier_info_plot, scaled_size_fng, leg = plot_glacier_locations_map(
    glacier_info=glacier_df_CEU,
    data_region=data_CEU,
    glacier_outline_rgi=glacier_outline_CEU,
    title="Glacier measurement locations Central European Alps",
    extent=(5.8, 15, 44, 48),  # (lon W, lon E, lat S, lat N)
    use_size=False,
    fixed_size=300,
    figsize=(10, 8),
    hue_col="SOURCE_CODE",
    hue_legend_title="Country",
    palette=palette_ceu,
    legend_separate=True)
# save figure: map and legend separately (combined into Fig. 1e outside the code)
fig.savefig('figures/paperTF/fig1_e_maps_CEU_gl.pdf')
leg.savefig("figures/paperTF/fig1_e_maps_CEU_gl_legend.pdf",
            dpi=300,
            bbox_inches="tight")

### (c) Norway (NOR), Fig. 1c:

In [ ]:
# Only the Norwegian stakes of RGI 08 (SOURCE_CODE 'NOR') are loaded
data_NOR, glacier_outline_rgi, glacier_df_NOR = build_region_glacier_info(
    cfg,
    rgi_region_id="08",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_locations_map(
    glacier_info=glacier_df_NOR,
    data_region=data_NOR,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Norway",
    extent=(4, 24, 57, 71),  # (lon W, lon E, lat S, lat N)
    use_size=False,
    fixed_size=200,
    figsize=(10, 8),
    legend_separate=False,
    gridline_label_fontsize=10)

# save figure (Fig. 1c)
fig.savefig('figures/paperTF/fig1_c_maps_NOR_gl.pdf')

### (d) Iceland, Fig. 1d:

In [ ]:
data_ISL, glacier_outline_rgi, glacier_df_ISL = build_region_glacier_info(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_locations_map(
    glacier_info=glacier_df_ISL,
    data_region=data_ISL,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Iceland",
    extent=(-25, -12, 63, 67),  # (lon W, lon E, lat S, lat N)
    figsize=(10, 8),
    use_size=False,
    fixed_size=300,
    legend_separate=False)

# save figure (Fig. 1d)
fig.savefig('figures/paperTF/fig1_d_maps_ISL_gl.pdf')

### (b) Svalbard, Fig. 1b:

In [ ]:
data_SJM, glacier_outline_rgi, glacier_df_SJM = build_region_glacier_info(
    cfg,
    rgi_region_id="07",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_07_Svalbard/07_rgi60_Svalbard.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_locations_map(
    glacier_info=glacier_df_SJM,
    data_region=data_SJM,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Svalbard",
    extent=(10, 21, 76.5, 80),  # (lon W, lon E, lat S, lat N)
    figsize=(18, 10),
    use_size=False,
    fixed_size=500,
    legend_separate=False)

# save figure (Fig. 1b)
fig.savefig('figures/paperTF/fig1_b_maps_SJM_gl.pdf')

### (a) Alaska, Fig. 1a:

In [ ]:
data_ALA, glacier_outline_rgi, glacier_df_ALA = build_region_glacier_info(
    cfg,
    rgi_region_id="01",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_locations_map(
    glacier_info=glacier_df_ALA,
    data_region=data_ALA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Alaska",
    extent=(-170, -130, 55, 67),  # (lon W, lon E, lat S, lat N)
    figsize=(10, 8),
    use_size=False,
    fixed_size=350,
    legend_separate=False)

# save figure (Fig. 1a)
fig.savefig('figures/paperTF/fig1_a_maps_ALA_gl.pdf')

### (f) Central Asia, Fig. 1f:
Glaciers coloured by RGI sub-region (1–4). The legend is saved separately.

In [ ]:
data_CA, glacier_outline_rgi, glacier_df_CA = build_region_glacier_info(
    cfg,
    rgi_region_id="13",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_13_CentralAsia/13_rgi60_CentralAsia.shp",
)

# Colour per RGI sub-region (O2Region 1-4)
palette_ca = {
    "1": NATURE_PALETTE["orange"],
    "2": NATURE_PALETTE["vermillion"],
    "3": NATURE_PALETTE["bluish_green"],
    "4": NATURE_PALETTE["reddish_purple"],
}

fig, ax, glacier_info_plot, scaled_size_fn, legend_fig = plot_glacier_locations_map(
    glacier_info=glacier_df_CA,
    data_region=data_CA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central Asia",
    extent=(67, 90, 35, 46),  # (lon W, lon E, lat S, lat N)
    figsize=(10, 8),
    hue_col='O2Region',
    hue_legend_title='RGI sub-region',
    legend_loc='lower right',
    palette=palette_ca,
    use_size=False,
    fixed_size=300,
    gridline_label_fontsize=8,
    legend_separate=True)

# save figure: map and legend separately (combined into Fig. 1f outside the code)
fig.savefig('figures/paperTF/fig1_f_maps_CA_gl.pdf')
legend_fig.savefig("figures/paperTF/fig1_f_maps_CA_legend_color.pdf",
                   dpi=300,
                   bbox_inches="tight")

### Appendix A3 (b): test glaciers of Central Asia sub-region 3
Map of the five test glaciers of Central Asia sub-region 3, in the same colours as the scatter plot of Fig. A3a (`2.2 Within_region`). The glacier names are those of the glacier-level split made in `2.2 Within_region`; update them if that split changes.

In [ ]:
# The five test glaciers of the glacier-level (Sinkhorn) split of Central Asia
# sub-region 3 ('CA_3' in '2.2 Within_region'; cached as test_glaciers in
# WITHIN_REGION/within_region_splits_checkpoint_gl.pkl), with the same colours
# as in Fig. A3a
color_by_glacier_ca3 = {
    'BARKRAK SREDNIY': '#000000',
    'BARKRAK SREDNIY EAST BRANCH': '#e69f00',
    'GLACIER NO. 354 (AKSHIYRAK)': '#56b4e9',
    'GLACIER NO. 599 (KJUNGEI ALA-TOO)': '#009e73',
    'TURGEN-AKSUU': '#f0e442'
}
ca3_glacier_names = set(color_by_glacier_ca3)

# Per-glacier info and stakes of the test glaciers (glacier_df_CA is indexed by GLACIER)
ca3_glaciers = glacier_df_CA.loc[glacier_df_CA.index.isin(
    ca3_glacier_names)].copy()
ca3_glaciers["GLACIER"] = ca3_glaciers.index
data_ca3 = data_CA[data_CA["GLACIER"].isin(ca3_glacier_names)].copy()

missing_in_map = ca3_glacier_names - set(ca3_glaciers["GLACIER"].unique())
if missing_in_map:
    print(
        f"  ⚠ {len(missing_in_map)} glacier(s) from the pred-vs-truth plot not "
        f"found in glacier_df_CA (naming mismatch?): {missing_in_map}")

# glacier_outline_rgi is the Central Asia outline loaded in the Fig. 1f cell above,
# so run that cell first. The legend is shared with Fig. A3a and not saved here.
fig_map, ax_map, _, _, _ = plot_glacier_locations_map(
    glacier_info=ca3_glaciers,
    data_region=data_ca3,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Test glacier measurement locations — Central Asia 3",
    extent=(70, 80, 39, 44),
    figsize=(8, 8),
    hue_col="GLACIER",
    hue_legend_title="Glacier",
    palette=color_by_glacier_ca3,
    use_size=False,
    fixed_size=300,
    gridline_label_fontsize=NATURE_SPECS["font_min_pt"],
    legend_separate=True,
)

# save figure (Fig. A3b)
fig_map.savefig('figures/paperTF/appendix/A3_b.png')